Install packages


In [1]:
%pip install -qU langchain langchain-core langchain-groq langchain-huggingface langchain-pinecone pinecone sentence-transformers pymupdf groq pandas tabulate pillow python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 Imports

In [2]:
import os 
import io 
import time 
import base64
from pathlib import Path
from getpass import getpass



import fitz
import pandas as pd 

from PIL import Image
from IPython.display import display

from groq import Groq
from pinecone import Pinecone, ServerlessSpec

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate   
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore


d:\Multimodal_Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF and model configuration

In [3]:
from pathlib import Path

PDF_PATH = Path("NovaCore_Multimodal_Company_Report_2026.pdf")

# Fallback for Windows environment
if not PDF_PATH.exists():
    fallback = Path.cwd() / "NovaCore_Multimodal_Company_Report_2026.pdf"
    
    if fallback.exists():
        PDF_PATH = fallback

assert PDF_PATH.exists(), (
    "PDF not found. Keep NovaCore_Multimodal_Company_Report_2026.pdf "
    "in the same folder as this notebook."
)

print("PDF found at:", PDF_PATH)

PDF found at: NovaCore_Multimodal_Company_Report_2026.pdf


In [4]:


TEXT_MODEL = "openai/gpt-oss-20b"
VISION_MODEL = "qwen/qwen3.8-27b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PINECONE_INDEX_NAME = "novacore-multimodal-rag"
PINECONE_NAMESPACE = "fy2026-demo"

IMAGE_DIR = Path("novacore_extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

print("PDF:", PDF_PATH)
print("Pinecone index:", PINECONE_INDEX_NAME)
print("Pinecone namespace:", PINECONE_NAMESPACE)

PDF: NovaCore_Multimodal_Company_Report_2026.pdf
Pinecone index: novacore-multimodal-rag
Pinecone namespace: fy2026-demo


 Add Groq and Pinecone API keys


In [5]:

from dotenv import load_dotenv


# 1. Load environment variables from the .env file
load_dotenv()

# 2. Check and validate that keys are present
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is missing! Make sure it is defined in your .env file.")

if not os.getenv("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY is missing! Make sure it is defined in your .env file.")

# 3. Initialize your API clients
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

text_llm = ChatGroq(
    model=TEXT_MODEL,
    temperature=0,
)

print("API clients are ready.")

API clients are ready.


Helper: convert image to base64

In [6]:
def image_to_data_uri(image_path):
    image_path = Path(image_path)

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((1600,1600))

        buffer = io.BytesIO()
        img.save(buffer,format="JPEG", quality=85)

        encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{encoded}"


Helper: summarize image , charts and diagrams

image/ chart
     |

vision model
     |

text summary 
     | 

embeddings  
     | 

Pinecone


   
  

In [7]:
def summarize_visual(image_path, page_number):
    image_data = image_to_data_uri(image_path)

    prompt = f"""
This visual was extracted from page {page_number} of the NovaCore FY2026 Company Report.

Describe the useful business information visible in the visual.

If it is a chart or graph:
- Mention important values.
- Mention the highest and lowest values.
- Mention the main trend.

If it is a diagram:
- Identify important components.
- Explain the flow or relationship.

If it is a normal business image:
- Describe the useful factual information.

Keep the description concise and factual.
"""

    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_data
                        }
                    }
                ]
            }
        ],
        temperature=0,
        max_completion_tokens=500,
    )

    return response.choices[0].message.content.strip()

Extract text,table and visuals from PDF.

In [8]:
def extract_multimodal_documents(pdf_path):
    pdf = fitz.open(str(pdf_path))

    documents = []
    extracted_xrefs = set()

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1

        print(f"Processing page {page_number}.....")

        # ------------------------------------------------------
        # 1. TEXT
        # ------------------------------------------------------

        text = page.get_text("text").strip()

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_number,
                        "modality": "text",
                        "source": pdf_path.name,
                    },
                )
            )

        # ------------------------------------------------------
        # 2. TABLES
        # ------------------------------------------------------

        try:
            tables = page.find_tables().tables

            for table_number, table in enumerate(tables, start=1):
                df = table.to_pandas()

                if not df.empty:
                    table_text = df.to_markdown(index=False)

                    documents.append(
                        Document(
                            page_content=table_text,
                            metadata={
                                "page": page_number,
                                "modality": "table",
                                "table_number": table_number,
                                "source": pdf_path.name,
                            },
                        )
                    )

        except Exception as error:
            print("Table extraction warning:", error)

        # ------------------------------------------------------
        # 3. VISUAL / IMAGES
        # ------------------------------------------------------

        for image_number, image_info in enumerate(
            page.get_images(full=True),
            start=1,
        ):
            xref = image_info[0]

            # Do not process an identical embedded image twice
            if xref in extracted_xrefs:
                continue

            extracted_xrefs.add(xref)

            try:
                image_data = pdf.extract_image(xref)

                image_bytes = image_data["image"]
                image_extension = image_data["ext"]

                image_path = (
                    IMAGE_DIR
                    / f"page_{page_number}_image{image_number}.{image_extension}"
                )

                image_path.write_bytes(image_bytes)

                # Generate AI summary of the image
                summary = summarize_visual(
                    image_path=image_path,
                    page_number=page_number,
                )

                documents.append(
                    Document(
                        page_content=summary,
                        metadata={
                            "page": page_number,
                            "modality": "visual",
                            "image_path": str(image_path),
                            "source": pdf_path.name,
                        },
                    )
                )

            except Exception as error:
                print(
                    f"Vision warning on page {page_number}:",
                    error,
                )

    pdf.close()

    return documents


# ------------------------------------------------------
# RUN DOCUMENT EXTRACTION
# ------------------------------------------------------

documents = extract_multimodal_documents(PDF_PATH)

print()
print("Total LangChain documents:", len(documents))

Processing page 1.....
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Processing page 2.....
Processing page 3.....
Processing page 4.....
Processing page 5.....
Processing page 6.....
Processing page 7.....
Processing page 8.....
Processing page 9.....

Total LangChain documents: 24


 Inspect Extracted Content 

In [9]:
rows = []

for doc in documents:
    rows.append(
        {
            "page": doc.metadata["page"],
            "modality": doc.metadata["modality"],
            "preview": doc.page_content[:140].replace("\n",""),
            
        }
    )
summary_df = pd.DataFrame(rows)  
display(summary_df)

print("Documents by modality: ")
display(summary_df["modality"].value_counts())

,page,modality,preview
0,1,text,NovaCore Systems Ltd. - Fictional company repo...
1,1,table,| Headquarters | Singapore | Emplo...
2,1,visual,Based on the visual from the NovaCore FY2026 C...
3,2,text,NovaCore Systems Ltd. - Fictional company repo...
4,2,table,| Priority | 2026 target ...
5,3,text,NovaCore Systems Ltd. - Fictional company repo...
6,3,table,| Metric | FY2025 | FY2026 | Cha...
7,3,visual,"Based on the visual provided, here is the desc..."
8,4,text,NovaCore Systems Ltd. - Fictional company repo...
9,4,table,| Region | 2026 revenue | YoY growth ...


Documents by modality: 


modality
text      9
table     8
visual    7
Name: count, dtype: int64

CREATE EMBEDDINGS

In [10]:
embeddings= HuggingFaceEmbeddings(
    model_name= EMBEDDING_MODEL,
    encode_kwargs= {"normalize_embeddings":True}
)
# Detect the exact vector dimension automatically.
EMBEDDING_DIMENSION = len(
    embeddings.embed_query("dimension check")
)

print("Embedding model:",EMBEDDING_MODEL)
print("Embedding dimension",EMBEDDING_DIMENSION)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3121.26it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension 384


CREATING PINECONE SERVERLESS INDEX

In [11]:
pc = Pinecone(
    api_keys= os.environ["PINECONE_API_KEY"]
)

if not pc.has_index(PINECONE_INDEX_NAME):
    print("Creating Pinecone index.....")

    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension = EMBEDDING_DIMENSION,
        metric = "cosine",
        spec = ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Wait until Pinecone reports that the index is ready.

    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        print("Waiting for Pinecone index.....")
        time.sleep(2)

    print("Pinecone index is created.") 

else:
    print("Pinecone index already exists") 

index = pc.Index(PINECONE_INDEX_NAME)

print("Connected to:",PINECONE_INDEX_NAME)




Pinecone index already exists
Connected to: novacore-multimodal-rag


CLEAN THE DEMO NAME NAMESPACE

In [12]:
try:    
    index.delete(
        delete_all=True,
        namespace = PINECONE_NAMESPACE,

    )
    print("Cleared namespace:" , PINECONE_NAMESPACE)

except Exception as error:
    print(
        "Namespace was probably empty. Continuing....",
        error,
    )    


Cleared namespace: fy2026-demo


STORE LANGCHAIN DOCUMENTS IN PINECONE

Lanchain Documents
       |
Hugging Face Embeddings
       |
PineconeVectorStore
       |
    Pinecone

In [13]:
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,

)

vectorstore.add_documents(documents)
print("Documents uploaded to Pinecone.")

Documents uploaded to Pinecone.


CREATE PINECONE RETRIEVER

In [14]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

print("Retriever is ready.")

Retriever is ready.


TEST PINECONE RETRIEVER

In [15]:
query = "Which region has the highest revenue growth?"

retrieved_docs = retriever.invoke(query)

for i , doc in enumerate(retrieved_docs,start=1):
    print("="*80)
    print(
        f"Result {i} "
        f"Page {doc.metadata.get('page')}|"
        f"Type:{doc.metadata.get('modality')}"

    )
    print(doc.page_content[:700])
    print()



Result 1 Page 4.0|Type:visual
Based on the visual provided, here is the description of the business information:

**Chart Type:** Bar Chart
**Title:** 2026 Year-over-Year Revenue Growth by Region

**Key Data Points:**
The chart displays the projected revenue growth percentages for four specific geographic regions for the fiscal year 2026:
*   **Europe:** 31%
*   **North America:** 27%
*   **Asia Pacific:** 24%
*   **Middle East:** 18%

**Analysis:**
*   **Highest Value:** Europe shows the highest projected growth at **31%**.
*   **Lowest Value:** The Middle East shows the lowest projected growth at **18%**.
*   **Main Trend:** The data indicates a descending trend in growth rates across the regions listed, with Europe le

Result 2 Page 4.0|Type:table
| Region        | 2026 revenue   | YoY growth   |   Enterprise customers | CSAT   |
|:--------------|:---------------|:-------------|-----------------------:|:-------|
| Europe        | $41.2M         | 31%          |                    11

CREATE THE LANGCHAIN TEXT RAG CHAIN

Question
   |
Pinecone Retriever
   |
Relevant Context
   |
Promt
   |
Text LLM
   |
Answer

In [16]:
rag_prompt=ChatPromptTemplate.from_template(
    """
You are a helpful assistant answering questions about the Novacore
Systems FY2026 company report.

Use ONLY the retrieved context below.

if the answer is not available  in the context , say:
"I could not find the that information in the report . "

Mention the page numbers when possible.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
)

text_rag_chain=(
    rag_prompt
    | text_llm
    | StrOutputParser()
)

print("Text RAG Chain is Ready.")

Text RAG Chain is Ready.


FORMAT RETREIVED CONTEXT

In [17]:
def format_context(retrieved_docs):
    parts=[]

    for doc in retrieved_docs:
        page = doc.metadata.get("page")
        modality = doc.metadata.get("modality")

        parts.append(
            f"[Page{page} | {modality.upper()}]\n"
            f"{doc.page_content}"
        )

    return "\n\n.join(parts)"


IF PINECONE RETRIEVES VISUAL EVIDENCE:

Pinecone retrieves visual summary
             |
Read image_path from metadata
            |
Load the original image 
            |
Question + test + image  
            |
       Vision model                         

In [18]:
def answer_with_vision(question, context, image_paths):
    image_paths = image_paths[:3]

    content = [
        {
            "type": "text",
            "text": f"""
You are answering questions about the NovaCore Systems FY2026 report.

Use ONLY the retrieved context and the attached retrieved visuals.

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}

Instructions:
- Answer factually.
- Use the attached visuals when relevant.
- Mention page numbers.
- If the information is missing, say you could not find it.
"""
        }
    ]

    # Add retrieved images
    for image_path in image_paths:
        content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": image_to_data_uri(image_path)
                }
            }
        )

    # Send question + context + images to vision model
    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": content
            }
        ],
        temperature=0,
        max_completion_tokens=900,
    )

    return response.choices[0].message.content.strip()

Complete multimodal RAG Function.

This is the complete application:

Question
   ↓
Pinecone Retriever
   ↓
Top 5 documents
   ↓
Any visual?
   ├── No  → Text RAG chain
   └── Yes → Vision LLM + retrieved images
   ↓
Final answer

In [19]:
def ask_rag(question, show_sources=True, show_images=False):

    # 1. Retrieve from Pinecone
    retrieved_docs = retriever.invoke(question)

    # 2. Create context
    context = format_context(retrieved_docs)

    # 3. Check whether Pinecone returned visuals
    image_paths = []

    for doc in retrieved_docs:
        if doc.metadata.get("modality") == "visual":
            image_path = doc.metadata.get("image_path")

            if image_path and Path(image_path).exists():
                image_paths.append(image_path)

    # 4. Generate the answer
    if image_paths:
        answer = answer_with_vision(
            question=question,
            context=context,
            image_paths=image_paths,
        )
    else:
        answer = text_rag_chain.invoke(
            {
                "context": context,
                "question": question,
            }
        )

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    if show_sources:
        print("\nPINECONE RETRIEVED SOURCES:")

        for i, doc in enumerate(retrieved_docs, start=1):
            print(
                f"{i}. Page {doc.metadata.get('page')} "
                f"| {doc.metadata.get('modality')}"
            )

    if show_images and image_paths:
        print("\nRETRIEVED VISUALS:")

        for image_path in image_paths:
            display(Image.open(image_path))

    return {
        "answer": answer,
        "documents": retrieved_docs,
        "image_paths": image_paths,
    }
     

In [20]:
ask_rag(
    "What does NovaCore Systems do and where is the company headquartered?"
)
     


QUESTION:
What does NovaCore Systems do and where is the company headquartered?

ANSWER:
I could not find the that information in the report.

PINECONE RETRIEVED SOURCES:
1. Page 6.0 | text
2. Page 1.0 | text
3. Page 8.0 | text
4. Page 2.0 | text
5. Page 5.0 | text


{'answer': 'I could not find the that information in the report.',
 'documents': [Document(id='6bd34455-9521-4dcb-a5a3-d3a07b96be71', metadata={'modality': 'text', 'page': 6.0, 'source': 'NovaCore_Multimodal_Company_Report_2026.pdf'}, page_content='NovaCore Systems Ltd. - Fictional company report created for a multimodal RAG demonstration\nPage 6\n5. Manufacturing & Supply Chain\nInformation-bearing diagram + facility image\nThe supply-chain diagram above contains operational details that are not repeated in full elsewhere. In particular, it\nidentifies the Quality Lab in Singapore as the critical control point before products move to regional distribution\nhubs.\nFigure 3. Penang Smart Assembly Facility. The embedded image carries facility utilization and solar contribution values that can be\nqueried by a multimodal system.'),
  Document(id='6b74fdee-c9a8-428d-b46b-392f6bdadabe', metadata={'modality': 'text', 'page': 1.0, 'source': 'NovaCore_Multimodal_Company_Report_2026.pdf'}, page